# 5장 코드 필사

### 코드 5-1 라이브러리 호출

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms   # 데이터 전처리를 위해 사용하는 라이브러리
from torch.utils.data import Dataset, DataLoader

### 코드 5-2 CPU 혹은 GPU 장치 확인

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

### 코드 5-3 fashion_mnist 데이터셋 내려받기

In [ ]:
train_dataset = torchvision.datasets.FashionMNIST("../chap05/data", download=True,
                transform=transforms.Compose([transforms.ToTensor()]))   # ①
test_dataset = torchvision.datasets.FashionMNIST("../chap05/data", download=True,
               train=False, transform=transforms.Compose([transforms.ToTensor()]))   # 앞에서 훈련 데이터셋을 내려받았다면 여기에서는 테스트 데이터셋을 내려받습니다.

### 코드 5-4 fashion_mnist 데이터를 데이터로더에 전달

In [ ]:
train_loader = torch.utils.data.DataLoader(train_dataset,
                                           batch_size=100)   # ①
test_loader = torch.utils.data.DataLoader(test_dataset,
                                          batch_size=100)

### 코드 5-5 분류에 사용될 클래스 정의

In [ ]:
labels_map = {0 : 'T-Shirt', 1 : 'Trouser', 2 : 'Pullover', 3 : 'Dress', 4 : 'Coat',
5 : 'Sandal', 6 : 'Shirt', 7 : 'Sneaker', 8 : 'Bag', 9 : 'Ankle Boot'}   # 열 개의 클래스

fig = plt.figure(figsize=(8,8));   # 출력할 이미지의 가로세로 길이로 단위는 inch
columns = 4;
rows = 5;
for i in range(1, columns*rows +1):
    img_xy = np.random.randint(len(train_dataset));   # ①
    img = train_dataset[img_xy][0][0,:,:]   # ②
    fig.add_subplot(rows, columns, i)
    plt.title(labels_map[train_dataset[img_xy][1]])
    plt.axis('off')
    plt.imshow(img, cmap='gray')
plt.show()   # 20개의 이미지 데이터를 시각적으로 표현

### 코드 5-6 심층 신경망 모델 생성

In [ ]:
class FashionDNN(nn.Module):
    def __init__(self):   # ①
        super(FashionDNN, self).__init__()
        self.fc1 = nn.Linear(in_features=784, out_features=256)   # ②
        self.drop = nn.Dropout(0.25)   # ③
        self.fc2 = nn.Linear(in_features=256, out_features=128)
        self.fc3 = nn.Linear(in_features=128, out_features=10)

    def forward(self, input_data):   # ④
        out = input_data.view(-1, 784)   # ⑤
        out = F.relu(self.fc1(out))   # ⑥
        out = self.drop(out)
        out = F.relu(self.fc2(out))
        out = self.fc3(out)
        return out

### 코드 5-7 심층 신경망에서 필요한 파라미터 정의

In [ ]:
learning_rate = 0.001;
model = FashionDNN();
model.to(device)

criterion = nn.CrossEntropyLoss();   # 분류 문제에서 사용하는 손실 함수
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate);   # ①
print(model)

### 코드 5-8 심층 신경망을 이용한 모델 학습

In [ ]:
num_epochs = 5
count = 0
loss_list = []   # ①
iteration_list = []
accuracy_list = []

predictions_list = []
labels_list = []

for epoch in range(num_epochs):
    for images, labels in train_loader:   # ②
        images, labels = images.to(device), labels.to(device)   # ③

        train = Variable(images.view(100, 1, 28, 28))   # ④
        labels = Variable(labels)

        outputs = model(train)   # 학습 데이터를 모델에 적용
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        count += 1

        if not (count % 50):   # count를 50으로 나누었을 때 나머지가 0이 아니라면 실행
            total = 0
            correct = 0
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                labels_list.append(labels)
                test = Variable(images.view(100, 1, 28, 28))
                outputs = model(test)
                predictions = torch.max(outputs, 1)[1].to(device)
                predictions_list.append(predictions)
                correct += (predictions == labels).sum()
                total += len(labels)

            accuracy = correct * 100 / total   # ⑤
            loss_list.append(loss.data)   # ①'
            iteration_list.append(count)
            accuracy_list.append(accuracy)

        if not (count % 500):
            print("Iteration: {}, Loss: {}, Accuracy: {}%".format(count, loss.data,
                  accuracy))

### 코드 5-9 합성곱 네트워크 생성

In [ ]:
class FashionCNN(nn.Module):
    def __init__(self):
        super(FashionCNN, self).__init__()
        self.layer1 = nn.Sequential(   # ①
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),   # ②
            nn.BatchNorm2d(32),   # ③
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)   # ④
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc1 = nn.Linear(in_features=64*6*6, out_features=600)   # ⑤
        self.drop = nn.Dropout2d(0.25)
        self.fc2 = nn.Linear(in_features=600, out_features=120)
        self.fc3 = nn.Linear(in_features=120, out_features=10)   # 마지막 계층의 out_features는 클래스 개수를 의미

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = out.view(out.size(0), -1)   # ⑥
        out = self.fc1(out)
        out = self.drop(out)
        out = self.fc2(out)
        out = self.fc3(out)
        return out

### 코드 5-10 합성곱 네트워크를 위한 파라미터 정의

In [ ]:
learning_rate = 0.001;
model = FashionCNN();
model.to(device)

criterion = nn.CrossEntropyLoss();
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate);
print(model)

### 코드 5-11 모델 학습 및 성능 평가

In [ ]:
num_epochs = 5
count = 0
loss_list = []
iteration_list = []
accuracy_list = []

predictions_list = []
labels_list = []

for epoch in range(num_epochs):
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        train = Variable(images.view(100, 1, 28, 28))
        labels = Variable(labels)

        outputs = model(train)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        count += 1

        if not (count % 50):
            total = 0
            correct = 0
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                labels_list.append(labels)
                test = Variable(images.view(100, 1, 28, 28))
                outputs = model(test)
                predictions = torch.max(outputs, 1)[1].to(device)
                predictions_list.append(predictions)
                correct += (predictions == labels).sum()
                total += len(labels)

            accuracy = correct * 100 / total
            loss_list.append(loss.data)
            iteration_list.append(count)
            accuracy_list.append(accuracy)

        if not (count % 500):
            print("Iteration: {}, Loss: {}, Accuracy: {}%".format(count, loss.data,
                  accuracy))